# Evaluation Notebook

Clones the repo (which already contains all adapters) and runs the full eval suite.

**Before running:**
1. Enable GPU: Notebook settings → Accelerator → **T4 x2** or **P100**
2. Enable Internet: Notebook settings → Internet → **On**
3. Add HuggingFace token: Add-ons → Secrets → **HF_TOKEN** (toggle On)

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HF login OK")

In [ ]:
REPO_URL = "https://github.com/202520030411/Distill-Safety-Aligned-Models.git"
REPO_DIR = "/kaggle/working/repo"

!git clone {REPO_URL} {REPO_DIR}

print("\nAdapters available:")
!ls {REPO_DIR}/outputs/

## Choose Models to Evaluate

Edit `MODELS_TO_EVAL` below to pick which models to run.

~5–8 min per 1B model, ~15–20 min for teacher (8B).

In [ ]:
MODELS_TO_EVAL = [
    "with_refusals",
    "dpo_with_refusals",
    "on_policy_dpo_with_refusals",
]

model_str = " ".join(MODELS_TO_EVAL)
print(f"Will evaluate: {model_str}")

In [ ]:
os.chdir(f"{REPO_DIR}/eval")
!python run_all.py --models {model_str}

## View Results

In [ ]:
import json

summary_path = f"{REPO_DIR}/results/eval/summary.json"
with open(summary_path) as f:
    summary = json.load(f)

print(f"{'Model':<35} {'Unsafe':>8} {'FalseRef':>8} {'Quality':>8} {'Jailbrk':>8}")
print("-" * 75)
for model, metrics in summary.items():
    print(f"{model:<35} "
          f"{metrics['unsafe_compliance_rate']:>8.3f} "
          f"{metrics['false_refusal_rate']:>8.3f} "
          f"{metrics['reference_quality_proxy']:>8.3f} "
          f"{metrics['jailbreak_unsafe_compliance_rate_macro']:>8.3f}")

## Download Results

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/eval_results", "zip",
                    f"{REPO_DIR}/results", "eval")
print("Download eval_results.zip from the Output tab.")